In [7]:

def generate_condprob(parent, states, mode="max_entropy", n_transitions=5):
    import numpy as np
    import random
    #append probabilities to each row in the condition table
    condprob=[]
        
    #for each parent state
    for parentstate in states:
        
        #subset all rows starting with parent state i
        subset = [row for row in parent if row[0] == parentstate]

        
        #All rows, starting with !, should lead only to !
        #If a sequence has ! at any point, every subsequent entry becomes !
        
        if mode=="min_entropy":
            raise ValueError('Minimum entropy not valid for a HOMC: min_ent is a deterministic process.')
        
        """
        if mode=="max_entropy":
            #get list of probabilities for each state
            #vec = np.random.random(len(subset))
            vec = np.ones(len(subset))
        """

        if mode=="max_entropy":
            raise ValueError('Max entropy not valid for a HOMC: max_ent is the same as for a memoryless process.')
        
        if mode=="med_entropy":
            #get n random rows with probability > 0, and 0 for rest of the rows
            vec = np.zeros(len(subset)).tolist()
            
            ids = list(range(0,len(vec)))
            selected = random.sample(ids, n_transitions)
            
            for i in selected:
                vec[i] = np.round(np.random.random(1)[0], decimals=8)

        #normalize it
        vec = np.round(vec/np.sum(vec), decimals=5)
        vec = vec.tolist()
        
        for i in range(0,len(subset)):
            #get the probability
            p = vec[i]
            
            #append it to row i in subset
            subset[i].append(p)
            
        #append to final list
        condprob = condprob + subset
    
    return condprob

In [8]:
import numpy as np

def GenerateHOMC(D = ["a","b","c","d"], # statespace for intial prob
                 mode = ["min_entropy","med_entropy","max_entropy"][0], # complexity
                 n_transitions = 2 # number of transitions with med_entropy
                 ):
    """
    Create a higher-order markov chain
    """
    #synbps.simulation.
    from homc_helpers import cartesian_product, combine_to_list, modify_rules, generate_condprob
    from alg2_initial_probabilities import GenerateInitialProb

    # A function to list-based conditional probabilities into a nested dictionary:
    def transform_markov_chain(markov_chain_list):
        markov_chain_dict = {}
        for table in markov_chain_list:
            order = len(table[0]) - 2
            transitions = {}
            for row in table:
                state, next_state, probability = tuple(row[:-2]), row[-2], row[-1]
                if state not in transitions:
                    transitions[state] = {}
                transitions[state][next_state] = probability
            markov_chain_dict[order] = transitions
        return markov_chain_dict
    
    # statespace for conditional prob
    D_abs = D.copy()
    D_abs.append("!")

    # Unconditional probabilities, excluding the absorbing state
    P0 = GenerateInitialProb(D=D, 
                            p0_type=mode)

    # copy for use in higher order probability tables
    states = D_abs.copy()

    ######################################
    # P1

    #for each link
    c = cartesian_product(states, states)
    d = combine_to_list(c)

    #final steps
    g = modify_rules(d, states)
    P1 = generate_condprob(g, states, mode, n_transitions)

    ######################################
    # P2

    #for each link
    c = cartesian_product(states, states)
    d = combine_to_list(c)

    e = cartesian_product(c, states)
    f = combine_to_list(e)

    #final steps
    g = modify_rules(f, states)
    P2 = generate_condprob(g, states, mode, n_transitions)

    ######################################

    # final probability tables
    HOMC = [P1, P2]

    # convert to dictionary
    P_k = transform_markov_chain(HOMC)

    return P0, P_k, HOMC




In [9]:
def SampleHOMC(D, P0, P_k):
    """
    Sample from the higher-order markov chain
    """

    # placeholder for trace
    sigma = []

    # stop when absorbing state is reached
    while "!" not in set(sigma):
        
        # determine tracelength
        tracelen = len(sigma)

        print(sigma)
        
        if tracelen == 0:
            #sample first event from P0
            e_t = np.random.choice(D,
                            size=1, 
                            replace=False, 
                            p=P0)[0]
            # add first event to trace
            sigma.append(e_t)
        
        # if length is less than or equal K-1, use K'th order probability table
        if tracelen > 0 and tracelen <= max(P_k.keys()):
            # retrieve the order to subset P_k from
            order = len(sigma)
        
            # retrieve the probability distribution
            prob_dist = P_k[order][tuple(sigma)]
        
            # Extract elements and their associated probabilities
            elements = list(prob_dist.keys())
            probabilities = list(prob_dist.values())
            
            # Use np.random.choice with the probabilities
            e_t = np.random.choice(elements,
                                    size=1, 
                                    replace=False, 
                                    p=probabilities)
            
            # add sampled event to trace
            sigma.append(e_t[0])
        
        # if length is > K-1, reach the absorbing state
        if tracelen > max(P_k.keys()):
            # truncate/end everything after K
            e_t = "!"
            
            # add sampled event to trace
            sigma.append(e_t)

    return sigma


In [10]:
"""
Deadlock

It appear a logic is needed that adds the absorbing state, in cases where probabilities doesnt sum to 1
OR
that there is an underlying problem, as probabilities doesnt sum to 1
"""


'\nminimum entropy: this is a first-order markov chain\n\nmedium + maximum entropy: in some situations, the process reaches a deadlock at step 2.\n\nIt appear a logic is needed that adds the absorbing state, in cases where probabilities doesnt sum to 1\nOR\nthat there is an underlying problem, as probabilities doesnt sum to 1\n'

In [14]:
#for i in range(0,10):
############ Testing
D = ["a","b","c","d"]

P0, P_k, HOMC = GenerateHOMC(D, # statespace for intial prob
                    mode = ["min_entropy","med_entropy","max_entropy"][0], # complexity
                    n_transitions = 4 # number of transitions with med_entropy
                    )

sigma = SampleHOMC(D, P0, P_k)
print(sigma)

ValueError: Minimum entropy not valid for a HOMC: This is a deterministic process.

In [12]:
P0

array([0., 0., 0., 1.])

In [13]:
P_k[1]#[("a",)]

{('a',): {'a': 0.33219, 'b': 0.24975, 'c': 0.12647, 'd': 0.0, '!': 0.29159},
 ('b',): {'a': 0.45643, 'b': 0.0, 'c': 0.29085, 'd': 0.01693, '!': 0.23579},
 ('c',): {'a': 0.47106, 'b': 0.16978, 'c': 0.20808, 'd': 0.15108, '!': 0.0},
 ('d',): {'a': 0.17215, 'b': 0.0, 'c': 0.24754, 'd': 0.53154, '!': 0.04878},
 ('!',): {'!': 0.20097}}

In [ ]:
P_k[2]

{('a', 'a'): {'a': 0.01711,
  'b': 0.05969,
  'c': 0.06663,
  'd': 0.00821,
  '!': 0.0191},
 ('a', 'b'): {'a': 0.06436,
  'b': 0.0058,
  'c': 0.06515,
  'd': 0.01425,
  '!': 0.04348},
 ('a', 'c'): {'a': 0.05578,
  'b': 0.04201,
  'c': 0.07089,
  'd': 0.03692,
  '!': 0.00506},
 ('a', 'd'): {'a': 0.041,
  'b': 0.04052,
  'c': 0.0631,
  'd': 0.06227,
  '!': 0.05302},
 ('a', '!'): {'!': 0.06045},
 ('b', 'a'): {'a': 0.07674,
  'b': 0.04303,
  'c': 0.08492,
  'd': 0.04569,
  '!': 0.02376},
 ('b', 'b'): {'a': 0.00326,
  'b': 0.0073,
  'c': 0.04033,
  'd': 0.05814,
  '!': 0.01},
 ('b', 'c'): {'a': 0.01695,
  'b': 0.08793,
  'c': 0.0792,
  'd': 0.04489,
  '!': 0.03278},
 ('b', 'd'): {'a': 0.01836,
  'b': 0.00094,
  'c': 0.05506,
  'd': 0.01007,
  '!': 0.05187},
 ('b', '!'): {'!': 0.08977},
 ('c', 'a'): {'a': 0.06531,
  'b': 0.05362,
  'c': 0.00358,
  'd': 0.05976,
  '!': 0.01418},
 ('c', 'b'): {'a': 0.05969,
  'b': 0.04088,
  'c': 0.05119,
  'd': 0.03549,
  '!': 0.01341},
 ('c', 'c'): {'a': 0.0

In [ ]:
HOMC[1]

[['a', 'a', 'a', 0.01711],
 ['a', 'a', 'b', 0.05969],
 ['a', 'a', 'c', 0.06663],
 ['a', 'a', 'd', 0.00821],
 ['a', 'a', '!', 0.0191],
 ['a', 'b', 'a', 0.06436],
 ['a', 'b', 'b', 0.0058],
 ['a', 'b', 'c', 0.06515],
 ['a', 'b', 'd', 0.01425],
 ['a', 'b', '!', 0.04348],
 ['a', 'c', 'a', 0.05578],
 ['a', 'c', 'b', 0.04201],
 ['a', 'c', 'c', 0.07089],
 ['a', 'c', 'd', 0.03692],
 ['a', 'c', '!', 0.00506],
 ['a', 'd', 'a', 0.041],
 ['a', 'd', 'b', 0.04052],
 ['a', 'd', 'c', 0.0631],
 ['a', 'd', 'd', 0.06227],
 ['a', 'd', '!', 0.05302],
 ['a', '!', '!', 0.00927],
 ['a', '!', '!', 0.01501],
 ['a', '!', '!', 0.0577],
 ['a', '!', '!', 0.02323],
 ['a', '!', '!', 0.06045],
 ['b', 'a', 'a', 0.07674],
 ['b', 'a', 'b', 0.04303],
 ['b', 'a', 'c', 0.08492],
 ['b', 'a', 'd', 0.04569],
 ['b', 'a', '!', 0.02376],
 ['b', 'b', 'a', 0.00326],
 ['b', 'b', 'b', 0.0073],
 ['b', 'b', 'c', 0.04033],
 ['b', 'b', 'd', 0.05814],
 ['b', 'b', '!', 0.01],
 ['b', 'c', 'a', 0.01695],
 ['b', 'c', 'b', 0.08793],
 ['b', 'c',